# B2-019 — Session 5: Transformer Blocks and Architecture

*90 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260808`  
**Qualified Book 1 prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C11-neural-training`  
**Remediation:** review the linked Book 1 units before continuing: [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb).


## 1. Position-wise feed-forward sublayer

The same two-layer MLP is applied independently to every token: $\operatorname{FFN}(x)=W_2\phi(W_1x+b_1)+b_2$. Typically $W_1$ expands width from $d$ to $d_{ff}$ and $W_2$ contracts back to $d$, allowing nonlinear feature transformation while preserving `(B,n,d)`. The weights are shared across positions, but different token rows generally produce different outputs.

**Worked example 1.** A `(2,3,4)` tensor passed through widths `4 -> 8 -> 4` remains `(2,3,4)`; changing one token row cannot directly change another row inside this sublayer.

**Checkpoint 1A.** Which dimensions are shared across tokens?

**Checkpoint 1B.** Why is this called position-wise?

In [ ]:
import torch
from torch import nn
SEED = 20260808
torch.manual_seed(SEED)
x = torch.arange(24, dtype=torch.float32).reshape(2, 3, 4)
ffn = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 4))
assert ffn(x).shape == x.shape

## 2. Pre-norm residual ordering

A pre-norm block computes $u=\operatorname{LN}_1(x)$, $y=x+\operatorname{MHA}(u;M)$, $v=\operatorname{LN}_2(y)$, then $z=y+\operatorname{FFN}(v)$. Each residual adds tensors of identical `(B,n,d)` shape. LayerNorm transforms features within each token row; the attention sublayer is the operation that mixes positions. Naming all four intermediates prevents accidentally normalizing the wrong residual source.

**Worked example 2.** If either sublayer returns zero, its residual path returns the incoming tensor exactly. Reversing to `LN(x + sublayer(x))` is a different post-norm architecture.

**Checkpoint 2A.** Which tensor is normalized before the second sublayer?

**Checkpoint 2B.** Where is the mask passed?

In [ ]:
def prenorm_trace(x, attention, ffn, norm1, norm2, mask):
    y = x + attention(norm1(x), mask=mask)
    z = y + ffn(norm2(y))
    return z

## 3. Encoder self-attention role

Let source states be $E\in\mathbb R^{B\times n_s\times d}$. Encoder self-attention derives Q, K, and V from E, so every output row updates one source position using source context. The usual mask is a source padding mask broadcast over query rows; causality is not required when the entire source is available. The encoder output keeps length $n_s$ and becomes memory for a decoder or features for an encoder-only head.

**Checkpoint 3A.** What are the Q/K/V source tensors in encoder self-attention?

**Checkpoint 3B.** Which mask excludes padded source keys?

## 4. Decoder causal self-attention role

Let current decoder states be $D\in\mathbb R^{B\times n_t\times d}$. Decoder self-attention derives Q, K, and V from D, but its allowed mask is the logical AND of the lower-triangular causal constraint and decoder key validity. This lets destination $i$ use target-side positions at or before $i$ while excluding padding. The output still has target length $n_t$.

**Worked example 3.** At target position 2 in a length-5 sequence, causal self-attention may use target positions `{0,1,2}` and must assign exactly zero weight to `{3,4}`.

**Checkpoint 4A.** Why is target padding alone insufficient?

**Checkpoint 4B.** Which target positions can row zero read?

## 5. Cross-attention role and dimensions

Cross-attention updates decoder destinations using encoder memory: Q comes from decoder states D, while K and V come from encoder outputs E. Therefore scores have shape `(B,h,n_t,n_s)` and outputs have shape `(B,n_t,d)`. Query length controls score rows; source-memory length controls columns. The mask normally excludes padded encoder keys and does not use the target causal triangle because source positions are not future target tokens.

**Worked example 4.** If decoder length is 4 and encoder length is 7, cross-attention scores have shape `(B,h,4,7)`. Changing K/V to decoder states would silently turn this back into self-attention.

**Checkpoint 5A.** Which length controls score rows?

**Checkpoint 5B.** Which length controls score columns?

## 6. Mask flow through an encoder-decoder layer

Keep three mask contracts distinct. The source padding mask enters encoder self-attention and later cross-attention as a key mask. The target causal mask combines with target key validity and enters decoder self-attention only. The position-wise FFN receives no attention mask. A useful mask flow ledger is: `source_valid -> encoder self-attention -> encoder memory`; `target_valid AND causal -> decoder self-attention`; `source_valid -> cross-attention`.

**Worked example 5.** If source position 6 is padding, every encoder query and every decoder cross-attention query must assign it zero weight. If target position 4 is future relative to query 2, only decoder causal self-attention blocks that edge.

**Checkpoint 6A.** Which mask is reused by cross-attention?

**Checkpoint 6B.** Why should the causal triangle not mask encoder-memory columns?

## 7. Architecture trace

An encoder stack repeatedly applies encoder self-attention and position-wise FFN blocks to produce contextual source memory. A decoder layer applies causal self-attention, then cross-attention to that memory, then a position-wise FFN, with its own norm/residual pair around every sublayer. A final projection may turn decoder states into class or vocabulary logits. This unit studies mechanics; larger language-model objectives begin in B2-020.

**Checkpoint 7A.** Does an encoder-only block require a causal mask by definition?

**Checkpoint 7B.** Which sublayer changes neither batch nor sequence length?

## 8. Common pitfalls and completion

**Common pitfalls.** Broken: normalize the residual branch after overwriting its source. Fix: name intermediate `y` and follow the pre-norm equations. Broken: feed encoder outputs into decoder queries. Fix: Q names the destinations being updated, so cross-attention queries come from the decoder. Broken: pass the causal target mask into cross-attention. Fix: trace the source padding mask and target causal mask separately.

**Exam connections.** Practice p22 follows only after this encoder/decoder/cross-attention role and mask flow: architecture questions reward explicit Q/K/V sources, exact score dimensions, correct mask ownership, and a norm-residual trace—not a memorized diagram.

**Going deeper.** `B2-020-language-transformers` applies these blocks to language objectives after this unit is complete.

Checkpoint answers: 1A weights and biases; 1B no token mixing; 2A y; 2B attention sublayer; 3A all three come from encoder states; 3B source key-valid mask; 4A it does not prevent future target leakage; 4B position zero only; 5A target query length; 5B encoder source length; 6A source padding/key-valid mask; 6B encoder positions are context, not future target positions; 7A no; 7B position-wise FFN.